# 04 — Klasifikimi dhe regresioni logjistik

        **Pyetja udhëheqëse:** Si dallojmë sinjalin nga sfondi dhe si zgjedhim pragun e vendimit?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të interpretohet regresioni logjistik probabilistikisht.
- Të implementohet sigmoid + cross-entropy.
- Të përdoren confusion matrix, ROC, AUC dhe precision-recall.
- Të kuptohet roli i pragut.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

## Pse kjo temë është e nevojshme?

Shumë probleme në fizikën eksperimentale nuk kërkojnë një numër real, por një vendim: sinjal apo sfond, ngjarje e vlefshme apo artefakt, regjim A apo regjim B. Regresioni logjistik na jep jo vetëm klasën, por një vlerësim probabilistik që mund të krahasohet me kostot shkencore të gabimeve.

Pika kryesore është se klasifikimi nuk mbaron te accuracy. Duhet të kuptojmë pragun, matricën e konfuzionit, ROC, precision–recall dhe faktin se vendimi optimal varet nga pyetja fizike.

### Si duhet lexuar ky notebook

Për çdo bllok kodi do të shpjegojmë **pse** kryhet ai hap, çfarë pyetjeje i përgjigjet dhe çfarë duhet të kontrollojmë në dalje. Kodi nuk trajtohet si recetë: çdo grafik dhe çdo metrikë duhet të rikthehet te hipoteza matematikore dhe te interpretimi fizik. Kur një rezultat nuk sillet siç presim, pyetja e parë nuk është “si ta rrisim score-in?”, por “cilën hipotezë të modelit ose të të dhënave po shkelim?”.

### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("04 — Klasifikimi dhe regresioni logjistik")
print("Seed:", RANDOM_STATE)

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 1. Sinjal kundrejt sfondit në një detektor — CORE

Regresioni logjistik modelon
$$
P(y=1|\mathbf x)=\sigma(\mathbf w^T\mathbf x+b),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
$$
Klasifikimi final kërkon një prag $t$: deklarojmë sinjal nëse $P(y=1|x)\ge t$.

### Para se të ekzekutojmë: Ndërtimi i problemit sinjal–sfond

Krijojmë dy popullata me mbivendosje të kontrolluar. Mbivendosja është thelbësore: në një problem trivial ku klasat ndahen plotësisht, nuk do të kishim nevojë të diskutonim pragje, ROC ose kompromis ndërmjet llojeve të gabimeve.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Ndërtimi i problemit sinjal–sfond.
n0, n1 = 500, 260
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
X0 = rng.normal([0.0, 0.0], [1.0, 1.1], size=(n0,2))
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
X1 = rng.normal([1.8, 1.5], [0.9, 0.8], size=(n1,2))
X = np.vstack([X0, X1]); y = np.r_[np.zeros(n0), np.ones(n1)]

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.scatter(X0[:,0], X0[:,1], s=10, label="sfond")
ax.scatter(X1[:,0], X1[:,1], s=10, label="sinjal")
ax.set_xlabel("Veçoria 1"); ax.set_ylabel("Veçoria 2"); ax.legend()
ax.set_title("Ngjarje detektori")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 2. Sigmoid dhe funksioni i humbjes — CORE

### Para se të ekzekutojmë: Nga score linear te probabiliteti

Vizualizojmë sigmoidin për të kuptuar si një score i pakufizuar shndërrohet në vlerë midis 0 dhe 1. Zona rreth $z=0$ është ajo ku modeli është më i pasigurt; larg saj probabiliteti saturon.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Nga score linear te probabiliteti.
z = np.linspace(-8, 8, 400)
sigmoid = lambda z: 1/(1+np.exp(-z))
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.plot(z, sigmoid(z))
ax.axhline(0.5, linestyle="--"); ax.axvline(0, linestyle="--")
ax.set_xlabel("z = w^T x + b"); ax.set_ylabel("σ(z)")
ax.set_title("Funksioni sigmoid")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

Për një mostër binare, entropia e kryqëzuar (cross-entropy) është
$$
\ell=-\left[y\log p+(1-y)\log(1-p)\right].
$$
Ajo penalizon fort parashikimet shumë të sigurta por të gabuara.

# 3. Zbatim manual me zbritje gradienti — CORE

### Para se të ekzekutojmë: Trajnimi manual i regresionit logjistik

Standardizojmë veçoritë dhe minimizojmë cross-entropy me zbritje gradienti. Historia e humbjes është diagnostikë e optimizimit: para se të besojmë metrikat e klasifikimit, duhet të dimë nëse procedura e trajnimit ka konverguar.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Trajnimi manual i regresionit logjistik.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.preprocessing import StandardScaler
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.model_selection import train_test_split

# Ndajmë të dhënat në mënyrë të riprodhueshme; test-i nuk përdoret për trajnim.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
# Përshtatim standardizimin vetëm mbi të dhënat e trajnimit për të shmangur leakage.
sc = StandardScaler().fit(Xtr)
Xs, Xts = sc.transform(Xtr), sc.transform(Xte)

A = np.c_[np.ones(len(Xs)), Xs]
w = np.zeros(A.shape[1])
loss_hist = []
lr = 0.15
# Përsërisim procedurën për të studiuar konvergjencën ose ndjeshmërinë ndaj parametrave.
for _ in range(1000):
    p = sigmoid(A@w)
    grad = A.T@(p-ytr)/len(ytr)
    w -= lr*grad
    eps=1e-12
    loss_hist.append(-(ytr*np.log(p+eps)+(1-ytr)*np.log(1-p+eps)).mean())

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.plot(loss_hist)
ax.set_xlabel("Iteracioni"); ax.set_ylabel("Cross-entropy")
ax.set_title("Konvergjenca e trajnimit")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

### Krahasim me implementimin kanonik në `src/`

### Para se të ekzekutojmë: Kontrolli me implementimin kanonik

Importojmë të njëjtin algoritëm nga `src/` për të verifikuar konsistencën. Ky është kontroll software-ik dhe pedagogjik: implementimi manual na mëson mekanizmin, ndërsa moduli i ripërdorshëm na mëson organizimin e kodit.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Kontrolli me implementimin kanonik.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import sys
# Kërkojmë automatikisht rrënjën e repository-t, në mënyrë që notebook-u të funksionojë nga dosje të ndryshme.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from src.classification.logistic import LogisticRegressionGD
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
log_src = LogisticRegressionGD(lr=0.15, n_iter=1000).fit(Xs, ytr)
print("Peshat nga src:", np.round(log_src.weights_, 4))

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 4. Krahasim me scikit-learn dhe kufiri i vendimit — CORE

### Para se të ekzekutojmë: Kufiri probabilistik i vendimit

Pasi trajnojmë versionin profesional, vizualizojmë vijën ku $P(y=1)=0.5$. Kjo vijë nuk është e vetmja zgjedhje e mundshme; ajo thjesht korrespondon me pragun standard 0.5.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Kufiri probabilistik i vendimit.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.linear_model import LogisticRegression
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
clf = LogisticRegression().fit(Xs, ytr)
# Llogarisim parashikimet mbi të dhëna që nuk përdoren për të ndryshuar parametrat e modelit.
proba = clf.predict_proba(Xts)[:,1]

xx, yy = np.meshgrid(np.linspace(Xs[:,0].min()-1, Xs[:,0].max()+1, 250),
                     np.linspace(Xs[:,1].min()-1, Xs[:,1].max()+1, 250))
# Llogarisim parashikimet mbi të dhëna që nuk përdoren për të ndryshuar parametrat e modelit.
P = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:,1].reshape(xx.shape)

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.contour(xx, yy, P, levels=[0.5])
ax.scatter(Xs[ytr==0,0], Xs[ytr==0,1], s=8)
ax.scatter(Xs[ytr==1,0], Xs[ytr==1,1], s=8)
ax.set_title("Kufiri i vendimit për t=0.5")
ax.set_xlabel("x1 i standardizuar"); ax.set_ylabel("x2 i standardizuar")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 5. Confusion matrix dhe pragu — CORE

### Para se të ekzekutojmë: Çfarë lloj gabimi po bëjmë?

Matrica e konfuzionit ndan gabimet në false positives dhe false negatives. Duke ndryshuar pragun, ne zhvendosim qëllimisht balancën ndërmjet tyre. Kjo është pika ku kriteri statistikor lidhet me koston fizike të një vendimi.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Çfarë lloj gabimi po bëjmë?.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
for t in [0.2, 0.5, 0.8]:
    pred=(proba>=t).astype(int)
    print(t, "precision=", round(precision_score(yte,pred),3),
          "recall=", round(recall_score(yte,pred),3),
          "F1=", round(f1_score(yte,pred),3))
ConfusionMatrixDisplay.from_predictions(yte, (proba>=0.5).astype(int))
plt.title("Matrica e konfuzionit, t=0.5")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 6. ROC, AUC dhe precision-recall — CORE

### Para se të ekzekutojmë: Vlerësimi në të gjitha pragjet

ROC dhe precision–recall nuk fiksohen në një prag të vetëm. Ato tregojnë si ndryshon sjellja e klasifikuesit kur lëvizim pragun. AUC mat aftësinë e renditjes, jo domosdoshmërisht cilësinë e kalibrimit probabilistik.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Vlerësimi në të gjitha pragjet.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
# Llogarisim TPR dhe FPR për të gjitha pragjet e mundshme.
fpr,tpr,_ = roc_curve(yte,proba)
auc=roc_auc_score(yte,proba)
prec,rec,_ = precision_recall_curve(yte,proba)

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.plot(fpr,tpr,label=f"AUC={auc:.3f}")
ax.plot([0,1],[0,1],linestyle="--")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.legend(); ax.set_title("Kurba ROC")
plt.show()

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.plot(rec,prec)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Kurba precision–recall")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

**Interpretim.** ROC përmbledh kompromisin TPR–FPR në të gjitha pragjet. Precision–recall bëhet veçanërisht i vlefshëm kur klasa pozitive është e rrallë.

Në një analizë reale ky interpretim duhet të shoqërohet me kontroll të njësive, madhësive karakteristike dhe hipotezave që lejojnë kalimin nga një tregues numerik te një përfundim fizik.

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 7. Si ndryshon pragu optimal? — EXTENDED

### Para se të ekzekutojmë: Zgjedhja e pragut si problem shkencor

Vizualizojmë precision, recall dhe F1 si funksion të pragut. Kjo e bën të qartë se “pragu optimal” ekziston vetëm pasi kemi përcaktuar se çfarë vlerësojmë dhe çfarë kostoje kanë gabimet.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Zgjedhja e pragut si problem shkencor.
thresholds=np.linspace(0.05,0.95,50)
Ps,Rs,F1s=[],[],[]
for t in thresholds:
    pred=(proba>=t).astype(int)
    Ps.append(precision_score(yte,pred,zero_division=0))
    Rs.append(recall_score(yte,pred,zero_division=0))
    F1s.append(f1_score(yte,pred,zero_division=0))
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.plot(thresholds,Ps,label="precision")
ax.plot(thresholds,Rs,label="recall")
ax.plot(thresholds,F1s,label="F1")
ax.set_xlabel("Pragu"); ax.legend(); ax.set_title("Pragu ndryshon kompromisin")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 8. Aplikim jo-fizik: diagnozë mjekësore — EXTENDED

Në screening mjekësor kostoja e një false negative mund të jetë shumë më e lartë se ajo e një false positive; prandaj pragu nuk duhet zgjedhur vetëm për të maksimizuar accuracy.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Llogaritni specificity për pragjet 0.2, 0.5 dhe 0.8.
2. Gjeni pragun që jep recall ≥ 0.95 me precision maksimal.
3. Shpjegoni interpretimin probabilistik të AUC.
4. Krahasoni ROC me precision-recall për një klasë pozitive 5%.

        ## B. Ushtrim i plotë
        Krijoni një problem të pabalancuar 95:5, trajnojeni regresionin logjistik dhe diskutoni pse accuracy mund të jetë mashtruese.

        ## C. Playground
        Ndryshoni distancën ndërmjet klasave, pabalancimin dhe pragun.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **Klasifikimi dhe regresioni logjistik** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.